Baixando Bibliotecas

In [1]:
%pip install -qU pypdf
%pip install -U langchain
%pip install -U langchain-community
%pip install -U langchain-groq
%pip install langchain-huggingface
%pip install langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 19.6 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.5.5
    Uninstalling langchain-core-1.5.5:
      Successfully uninstalled langchain-core-1.5.5
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.15
    Uninstalling langchain-1.3.15:
      Successfully uninstalled langchain-1.3.15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling

Configuração da LLM

In [2]:
from google.colab import userdata
import os

api_key = userdata.get('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = api_key

from langchain_groq import ChatGroq

llm = ChatGroq(model='openai/gpt-oss-120b')

In [3]:
prompt = '''
Explique para uma pessoa que nunca estudou Libras o que é a Língua Brasileira de Sinais
e por que ela é importante para a comunicação e inclusão de pessoas surdas.
'''

resposta = llm.invoke(prompt)

resposta

AIMessage(content='**O que é a Língua Brasileira de Sinais (Libras)?**  \n\n1. **Uma língua natural**  \n   - **Não é “gestos” ou “mímica”.** Assim como o português, o inglês ou o mandarim, a Libras tem sua própria gramática, sintaxe (ordem das palavras) e vocabulário.  \n   - **É visual‑espacial.** As informações são transmitidas por meio de **mãos, movimentos, expressões faciais e postura corporal**. Cada um desses elementos tem um significado preciso, e a combinação deles cria frases completas.  \n\n2. **Quem a usa**  \n   - A Libras é a primeira língua da comunidade surda no Brasil.  \n   - Mais de **10 milhões de brasileiros** têm algum grau de deficiência auditiva; cerca de **500 mil a 1 milhão** utilizam a Libras como principal meio de comunicação.  \n\n3. **Como funciona**  \n   - **Mãos:** Cada configuração (posição dos dedos, forma da mão) corresponde a um sinal.  \n   - **Movimento:** A direção, velocidade e trajeto do movimento podem mudar o sentido da palavra.  \n   - **Ex

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader1 = PyPDFLoader("/content/Curso Básico de Libras.pdf")

pages1 = []

for page in loader1.lazy_load():
    pages1.append(page)

print(f"Total de páginas do Curso Básico: {len(pages1)}")

/tmp/ipykernel_3107/2464440687.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Total de páginas do Curso Básico: 10


In [5]:
loader2 = PyPDFLoader("/content/fiocruz_libras.pdf")

pages2 = []

for page in loader2.lazy_load():
    pages2.append(page)

print(f"Total de páginas do Fiocruz: {len(pages2)}")

Total de páginas do Fiocruz: 15


In [6]:
all_pages = pages1 + pages2

print(f"Total de páginas nos dois PDFs: {len(all_pages)}")

Total de páginas nos dois PDFs: 25


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embed_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Criar o Vector Store

In [8]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore.from_documents(
    all_pages,
    embed_model
)

Criar o Retriever

In [9]:
retriever = vector_store.as_retriever()

Teste do RAG

In [10]:
docs = vector_store.similarity_search(
    "Qual é o objetivo de aprender Libras?",
    k=4
)

for doc in docs:
    print(f"Página: {doc.metadata.get('page')}")
    print(doc.page_content[:500])
    print("-" * 80)

Página: 8
17. PROGRAMA DA UNIDADE DIDÁTICA - PUD 
DISCIPLINA:  Básico de Libras 
Carga Horária: 40h  
Número de Créditos:    02 
Semestre: 2018.1  
EMENTA 
A disciplina Básico de Libras tem como objetivo levar os alunos a desenvolver habilidades comunicativas 
básicas em Libras com a finalidade de atender os preceitos de inclusão das pessoas surdas tanto no âmbito 
educacional como laboral determinado na Lei 10.436/02 e seu Decreto de regulamento 5.626/05 
atendendo as orientações que trata de sua difusão
--------------------------------------------------------------------------------
Página: 2
3
ACESSIBILIDADE E OS PRINCÍPIOS DO SUS
Sumário
1  SINAIS INTRODUTÓRIOS PARA O ATENDIMENTO EM SAÚDE:  
GRAMÁTICA DA LIBRAS: GRAMÁTICA DA LIBRAS (PARTE 1) 4
— 1.1 GRAMÁTICA DA LIBRAS  4
— 1.2 ALFABETO MANUAL DE LIBRAS 6
— 1.3 NÚMEROS 7
— 1.4 SISTEMA PRONOMINAL 8
— 1.5 SAUDAÇÕES, CUMPRIMENTOS E AGRADECIMENTOS 9
— 1.6 ASPECTOS MORFOLÓGICOS  10
— 1.7 RELAÇÕES DE GÊNERO 11
— 1.8 ASPECTOS SINTÁTICOS  

Template

In [11]:
from langchain_core.prompts import ChatPromptTemplate

template = """
Você é o LibrasIA, um assistente de IA especializado em ensinar
Libras para pessoas iniciantes.

Sua função é transformar as informações dos materiais de referência
em respostas claras, simples e objetivas.

REGRAS OBRIGATÓRIAS:
- Utilize exclusivamente as informações presentes no contexto.
- Resuma o conteúdo, nunca copie grandes trechos do material.
- Responda em no máximo 500 caracteres.
- Priorize somente as informações essenciais para responder à pergunta.
- Elimine espaços e quebras de linha desnecessários.
- Use linguagem simples e adequada para iniciantes.
- Não repita a pergunta na resposta.
- Não invente informações.
- Responda somente sobre o conteúdo dos dois PDFs.
- Prefira uma resposta curta em uma única frase ou pequeno parágrafo.
- Se a informação não estiver no contexto, responda:
"Não encontrei essa informação nos materiais consultados."

Contexto:
{context}

Pergunta:
{question}

Gere somente a resposta final, sem explicações adicionais.
"""

prompt = ChatPromptTemplate.from_template(template)

In [12]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [13]:
def limitar_resposta(texto: str) -> str:
    texto = " ".join(texto.split())

    if len(texto) <= 500:
        return texto

    return texto[:497].rsplit(" ", 1)[0] + "..."

In [14]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
    | limitar_resposta
)

In [15]:
response = chain.invoke(
    "Qual é o objetivo de aprender Libras?"
)

print(response)
print("\nQuantidade de caracteres:", len(response))

Aprender Libras visa promover a aquisição de noções básicas da língua, permitindo comunicação simples entre surdos e ouvintes, ampliar a inclusão e desenvolver práticas acessíveis em contextos educacionais e laborais.

Quantidade de caracteres: 217


Tool

In [16]:
from langchain_core.tools import tool

@tool
def pega_contexto(query: str) -> str:
    """Busca informações relevantes nos materiais de Libras."""

    resultado = retriever.invoke(query)

    return "\n\n".join(
        doc.page_content for doc in resultado
    )

tools = [pega_contexto]

In [17]:
print(pega_contexto.invoke("O que é Libras?"))

CURSO BÁSICO DE LIBRAS 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Fortaleza, 2019

17. PROGRAMA DA UNIDADE DIDÁTICA - PUD 
DISCIPLINA:  Básico de Libras 
Carga Horária: 40h  
Número de Créditos:    02 
Semestre: 2018.1  
EMENTA 
A disciplina Básico de Libras tem como objetivo levar os alunos a desenvolver habilidades comunicativas 
básicas em Libras com a finalidade de atender os preceitos de inclusão das pessoas surdas tanto no âmbito 
educacional como laboral determinado na Lei 10.436/02 e seu Decreto de regulamento 5.626/05 
atendendo as orientações que trata de sua difusão. A disciplina também abrange os conteúdos relacionados 
aos fundamentos históricos culturais da Libras e sua relação com a educação dos Surdos; Cultura e 
Identidade Surdas; Expressões não manuais; Uso do Espaço. Vocabulário da Libras em diferentes 
contextos. 
 
OBJETIVO 
Geral: Promover o aprendizado das noções básicas da Língua Brasileira de Sinais. 
Específicos: Divulgar a L IBRAS por meio de novos vocábulos e situações co

Criando o Agente

In [18]:
system_prompt = """
Você é o LibrasIA, um assistente de IA especializado em ensinar
Libras para pessoas iniciantes.

Utilize a ferramenta pega_contexto para buscar informações nos
materiais de Libras antes de responder.

Utilize somente informações encontradas nos materiais consultados.

Responda somente o que foi perguntado, de forma clara, simples
e objetiva.

Resuma as informações encontradas e não copie trechos extensos
dos documentos.

A resposta deve ter no máximo 500 caracteres.

Não invente informações e não utilize conhecimento externo.

Se a informação não estiver nos materiais consultados, responda:
"Não encontrei essa informação nos materiais consultados."
"""

In [19]:
from langgraph.prebuilt import create_react_agent

In [20]:
agent_pdf = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt
)

/tmp/ipykernel_3107/720291031.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_pdf = create_react_agent(


In [21]:
response = agent_pdf.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Qual é o objetivo de aprender Libras?"
        }
    ]
})

In [22]:
print(response["messages"][-1].content)

O objetivo de aprender Libras é desenvolver habilidades comunicativas básicas que possibilitem a inclusão de pessoas surdas nos ambientes educacional e laboral, atendendo à Lei 10.436/02. Ao dominar sinais, o aprendiz pode estabelecer comunicação simples com surdos, ampliar o acesso à informação e promover a acessibilidade e a integração entre surdos e ouvintes.


In [23]:
response = agent_pdf.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Quais são alguns sinais básicos para cumprimentar uma pessoa?"
        }
    ]
})

print(response["messages"][-1].content)

Nos materiais consultados, os sinais básicos de saudação são:

- **Beijo**: mão próxima aos lábios, gesto que imita um beijo.  
- **Tchau**: mão aberta, movimento de despedida para fora.

Esses são os sinais de saudação e despedida apresentados nas figuras do capítulo “Saudações, cumprimentos e agradecimentos”.


In [24]:
response = agent_pdf.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Qual foi o primeiro computador criado pela NASA?"
        }
    ]
})

print(response["messages"][-1].content)

Não encontrei essa informação nos materiais consultados.


In [25]:
response = agent_pdf.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Qua a importancia em aprender libras?"
        }
    ]
})

resposta = response["messages"][-1].content

print(resposta)
print("\nQuantidade de caracteres:", len(resposta))

Aprender Libras é essencial para promover a inclusão de pessoas surdas, conforme a Lei 10.436/02, permitindo comunicação básica em ambientes educacionais e laborais. A disciplina de Libras visa divulgar a língua, ampliar a comunicação entre surdos e ouvintes, e desenvolver habilidades que favorecem o respeito à cultura e identidade surda, contribuindo para uma sociedade mais acessível e igualitária.

Quantidade de caracteres: 402


In [26]:
def perguntar_librasia(pergunta: str) -> str:
    response = agent_pdf.invoke({
        "messages": [
            {
                "role": "user",
                "content": pergunta
            }
        ]
    })

    return response["messages"][-1].content

In [27]:
resposta = perguntar_librasia("O que é Libras?")

print(resposta)
print("\nQuantidade de caracteres:", len(resposta))

Libras – Língua Brasileira de Sinais – é a língua de sinais oficial do Brasil, reconhecida pela Lei 10.436/02. Possui gramática, vocabulário e expressões não‑manuais próprias, permitindo comunicação entre surdos e ouvintes e atendendo aos preceitos de inclusão no âmbito educacional e laboral. 

Quantidade de caracteres: 294


In [28]:
resposta = perguntar_librasia("Qual é a história da NASA?")

print(resposta)

Não encontrei essa informação nos materiais consultados.


Instalar o FastAPI

In [29]:
%pip install -q fastapi uvicorn

In [46]:
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="LibrasIA")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:4200"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

In [31]:
class Pergunta(BaseModel):
    pergunta: str


@app.post("/perguntar")
def perguntar(dados: Pergunta):
    resposta = perguntar_librasia(dados.pergunta)

    return {
        "resposta": resposta
    }

In [32]:
app = FastAPI(title="LibrasIA")

In [33]:
class Pergunta(BaseModel):
    pergunta: str

Criando o endpoint

In [34]:
@app.post("/perguntar")
def perguntar(dados: Pergunta):
    resposta = perguntar_librasia(dados.pergunta)

    return {
        "resposta": resposta
    }

Testes

In [35]:
print(app.routes)

[Route(path='/openapi.json', name='openapi', methods=['GET', 'HEAD']), Route(path='/docs', name='swagger_ui_html', methods=['GET', 'HEAD']), Route(path='/docs/oauth2-redirect', name='swagger_ui_redirect', methods=['GET', 'HEAD']), Route(path='/redoc', name='redoc_html', methods=['GET', 'HEAD']), APIRoute(path='/perguntar', name='perguntar', methods=['POST'])]


In [36]:
%pip install -q nest-asyncio pyngrok

In [37]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()

In [38]:
import threading

def iniciar_servidor():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=iniciar_servidor)
thread.start()

In [39]:
import requests

teste = requests.get("http://127.0.0.1:8000/docs")

print("Status:", teste.status_code)

INFO:     127.0.0.1:39440 - "GET /docs HTTP/1.1" 200 OK
Status: 200


In [40]:
dados = {
    "pergunta": "O que é Libras?"
}

teste = requests.post(
    "http://127.0.0.1:8000/perguntar",
    json=dados
)

print("Status:", teste.status_code)
print("Resposta:", teste.json())

INFO:     127.0.0.1:39442 - "POST /perguntar HTTP/1.1" 200 OK
Status: 200
Resposta: {'resposta': 'Libras (Língua Brasileira de Sinais) é a língua de sinais oficial do Brasil, reconhecida pela Lei\u202f10.436/02 e seu regulamento (Decreto\u202f5.626/05). Ela permite a comunicação entre surdos e ouvintes por meio de sinais manuais, expressões faciais e uso do espaço, abrangendo aspectos históricos, culturais e identitários da comunidade surda. Também inclui o alfabeto manual para soletrar palavras sem sinal específico.'}


In [41]:
thread = threading.Thread(target=iniciar_servidor)
thread.start()

INFO:     Started server process [3107]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


In [42]:
dados = {
    "pergunta": "O que é Libras?"
}

teste = requests.post(
    "http://127.0.0.1:8000/perguntar",
    json=dados
)

print("Status:", teste.status_code)
print("Resposta:", teste.json())

INFO:     127.0.0.1:45400 - "POST /perguntar HTTP/1.1" 200 OK
Status: 200
Resposta: {'resposta': 'Libras é a Língua Brasileira de Sinais, a língua natural dos surdos no Brasil. Possui gramática própria, alfabeto manual e vocabulário específico, e é reconhecida legalmente pela Lei\u202f10.436/02 e seu regulamento (Decreto\u202f5.626/05) para garantir a inclusão em educação e trabalho. É usada para comunicação básica entre surdos e ouvintes.'}


In [43]:
from google.colab import userdata
from pyngrok import ngrok

ngrok_token = userdata.get("NGROK_AUTHTOKEN")

ngrok.set_auth_token(ngrok_token)

In [44]:
public_url = ngrok.connect(8000)

print(public_url)

NgrokTunnel: "https://putdown-spending-crib.ngrok-free.dev" -> "http://localhost:8000"


In [45]:
dados = {
    "pergunta": "O que é Libras?"
}

teste = requests.post(
    "https://putdown-spending-crib.ngrok-free.dev/perguntar",
    json=dados
)

print("Status:", teste.status_code)
print("Resposta:", teste.json())

INFO:     34.75.234.225:0 - "POST /perguntar HTTP/1.1" 200 OK
Status: 200
Resposta: {'resposta': 'Libras –\u202fLíngua Brasileira de Sinais – é a língua de sinais reconhecida legalmente no Brasil (Lei\u202f10.436/02). Ela permite a comunicação entre pessoas surdas e ouvintes, abrange aspectos históricos, culturais e gramaticais próprios, e inclui uso de sinais manuais, expressões faciais, corporais e o espaço. É a principal ferramenta de inclusão e acesso à informação para a comunidade surda.'}
